# Micro-assignment 4.4: retrieval over a movie-plot corpus

Same pattern as the class notebook (chunk, embed, FAISS, metadata filter, hybrid), on a **new** dataset: real movie plots (`data/movies.jsonl`), not the IRS tax corpus. Encoding is transformer-based (`all-MiniLM-L6-v2`), exactly as in class; the model downloads on first use, so these cells need internet.

## Setup

```
pip install sentence-transformers faiss-cpu rank-bm25 numpy
```

In [ ]:
import json, numpy as np
rows = [json.loads(l) for l in open("data/movies.jsonl")]
ids = [r["id"] for r in rows]
plots = [r["plot"] for r in rows]
print(len(rows), "movies loaded")

### 1. Chunk a page
A long page is split into overlapping word windows (this part runs offline).

In [ ]:
def chunk_text(text, size=20, overlap=5):
    words = text.split(); chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + size]))
        if i + size >= len(words): break
        i += size - overlap
    return chunks

page = " ".join(r["plot"] for r in rows[:3])   # first three plots as one page
pieces = chunk_text(page)
print("page words:", len(page.split()), "chunks:", len(pieces),
      "overlap ok:", pieces[0].split()[-5:] == pieces[1].split()[:5])
# -> page words: 71 | chunks: 5 | overlap ok: True

### 2. Embed with a transformer and search with FAISS

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")          # downloads once
emb = model.encode(plots, normalize_embeddings=True)      # one vector per plot
index = faiss.IndexFlatIP(emb.shape[1])                   # inner product = cosine
index.add(np.asarray(emb, dtype="float32"))

def search(query, k=3):
    qv = np.asarray(model.encode(query, normalize_embeddings=True), dtype="float32")[None, :]
    scores, idx = index.search(qv, k)
    return [(ids[i], round(float(s), 3)) for s, i in zip(scores[0], idx[0])]

print(search("a detective hunting artificial humans in a future city", 3))
# Expected (real model): blade-runner-1982 is the top hit, matched by meaning
# even though the query shares few exact words with the plot.

### 3. Metadata filter: pick the right version of a remake

In [ ]:
# "The Italian Job" exists twice (1969 and 2003). Over-fetch, then keep the year
# you want, exactly like the tax-year filter in the class corpus.
def search_filtered(query, year, k=3, pool=len(rows)):
    qv = np.asarray(model.encode(query, normalize_embeddings=True), dtype="float32")[None, :]
    scores, idx = index.search(qv, pool)
    hits = [(ids[i], rows[i]["year"]) for i in idx[0] if rows[i]["year"] == year]
    return hits[:k]

print("no filter:", [h[0] for h in search("gold heist with mini coopers", 4)])
print("year 2003:", search_filtered("gold heist with mini coopers", 2003, 3))
# Expected: without a filter both italian-job-1969 and italian-job-2003 appear;
# the year filter keeps only 2003, so italian-job-2003 is returned.

### 4. Hybrid: meaning plus keywords

In [ ]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([p.lower().split() for p in plots])   # keyword baseline

def minmax(x):
    x = np.array(x, float); span = x.max() - x.min()
    return (x - x.min()) / span if span > 0 else np.zeros_like(x)
def ranked(scores, k=3):
    return [ids[i] for i in np.argsort(scores)[::-1][:k]]

query = "the one about the dream heist"
key = bm25.get_scores(query.lower().split())
vec = emb @ model.encode(query, normalize_embeddings=True)   # cosine (normalized)
hybrid = 0.5 * minmax(vec) + 0.5 * minmax(key)
print("keyword only:", ranked(key))
print("hybrid      :", ranked(hybrid))
# keyword only (deterministic): italian-job-2003 first, the WRONG film, because it
# matches "heist". Expected with the real model: the dense score lifts inception-2010
# (a "heist inside dreams"), and the hybrid puts it on top.

### 5. Why meaning wins here (reasoning)

In [ ]:
print(search("the one about the dream heist", 3))
# Expected (real model): inception-2010 near the top, even though BM25 ranked it
# last. The dense embedding maps the paraphrase "dream heist" close to Inception by
# MEANING; keyword search cannot, because the query shares no distinctive words with
# the plot. This is exactly why the class uses transformer embeddings over TF-IDF/BM25.